In [1]:
%load_ext autoreload
%autoreload 2
import sys, json, time
from pathlib import Path
import numpy as np
import polars as pl

ROOT = Path.cwd()
while ROOT != ROOT.parent and not ((ROOT / 'src').is_dir() and (ROOT / 'configs').is_dir()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config
from src.features.prepare import META
cfg = load_config()
PROC, REPORTS = ROOT / 'data' / 'processed', ROOT / 'reports'
SEED = cfg['seed']

lf = pl.scan_parquet(PROC / 'model_table.parquet')
schema = lf.collect_schema()
FEATS_ALL = [c for c in schema.names() if c not in META]

# R3 mở rộng (28/10): proxy thời gian pmts_year_1139T + 12 *_yrs_ago
R3_EXT = ([c for c in FEATS_ALL if c.startswith('credit_bureau_a_2__pmts_year_1139T__')]
          + [c for c in FEATS_ALL if c.endswith('_yrs_ago')])
FEATS = [c for c in FEATS_ALL if c not in R3_EXT]
print('R3 mở rộng loại:', len(R3_EXT), '| còn:', len(FEATS))

# R6: Champion loại PSI(Valid so với cửa sổ Train) > 0.25
psi = pl.read_csv(REPORTS / 'psi_valid_vs_window.csv').filter(pl.col('feature').is_in(FEATS))
R6 = {w: psi.filter(pl.col(w) > 0.25)['feature'].to_list() for w in ['A', 'B']}
CHAMP_FEATS = {w: [c for c in FEATS if c not in R6[w]] for w in ['A', 'B']}
for w in ['A', 'B']:
    print(w, '| R6 loại:', len(R6[w]), '| vào optbinning:', len(CHAMP_FEATS[w]))

(REPORTS / 'champion_feature_lists.json').write_text(
    json.dumps({'R3_ext': R3_EXT, 'R6': R6, 'champion_feats': CHAMP_FEATS}, indent=1),
    encoding='utf-8')

import optbinning
print('optbinning', optbinning.__version__)

R3 mở rộng loại: 15 | còn: 628
A | R6 loại: 73 | vào optbinning: 555
B | R6 loại: 16 | vào optbinning: 612
optbinning 1.0.0


In [2]:
import os
from joblib import Parallel, delayed
from optbinning import OptimalBinning

WIN = {'A': (0, 43), 'B': (20, 43)}
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

def load_train(w):
    w0, w1 = WIN[w]
    return (lf.filter((pl.col('split') == 'train') & pl.col('WEEK_NUM').is_between(w0, w1))
              .select(['target'] + CHAMP_FEATS[w])
              .collect(engine='streaming'))

def col_arrays(d, cols):
    for c in cols:
        s = d.get_column(c)
        is_cat = s.dtype == pl.String
        yield c, (s.to_numpy() if is_cat else s.cast(pl.Float64).to_numpy()), is_cat

def fit_one(name, x, y, is_cat):
    t0 = time.time()
    ob = OptimalBinning(
        name=name, dtype='categorical' if is_cat else 'numerical', solver='cp',
        max_n_prebins=20, min_prebin_size=0.05, min_bin_size=0.05,
        monotonic_trend=None if is_cat else 'auto_asc_desc',
        cat_cutoff=0.01 if is_cat else None, time_limit=60,
    )
    ob.fit(x, y)
    bt = ob.binning_table.build()
    return name, ob, dict(
        feature=name, dtype='cat' if is_cat else 'num', status=ob.status,
        n_bins=len(bt) - 3,                      # trừ các dòng Special, Missing, Totals
        iv=float(bt.loc['Totals', 'IV']), seconds=round(time.time() - t0, 2),
    )

def fit_many(d, cols):
    y = d['target'].to_numpy()
    out = Parallel(n_jobs=N_JOBS)(delayed(fit_one)(c, x, y, cat) for c, x, cat in col_arrays(d, cols))
    return {n: ob for n, ob, _ in out}, pl.DataFrame([r for _, _, r in out])

trB = load_train('B')
print(trB.shape, f'{trB.estimated_size("gb"):.2f} GB', '| N_JOBS =', N_JOBS)

t0 = time.time()
_, test = fit_many(trB, CHAMP_FEATS['B'][:24])
sec = time.time() - t0
print(f'24 feature: {sec:.1f}s → ước tính B: {sec * 612 / 24 / 60:.1f} phút, '
      f'A: {sec * 555 / 24 / 60 * 841500 / 522720:.1f} phút')
print(test.sort('iv', descending=True))

(522720, 613) 1.24 GB | N_JOBS = 11
24 feature: 14.3s → ước tính B: 6.1 phút, A: 8.9 phút
shape: (24, 6)
┌────────────────────────────────┬───────┬─────────┬────────┬──────────┬─────────┐
│ feature                        ┆ dtype ┆ status  ┆ n_bins ┆ iv       ┆ seconds │
│ ---                            ┆ ---   ┆ ---     ┆ ---    ┆ ---      ┆ ---     │
│ str                            ┆ str   ┆ str     ┆ i64    ┆ f64      ┆ f64     │
╞════════════════════════════════╪═══════╪═════════╪════════╪══════════╪═════════╡
│ avgdpdtolclosure24_3658938P    ┆ num   ┆ OPTIMAL ┆ 4      ┆ 0.330594 ┆ 0.48    │
│ avgdbddpdlast24m_3658932P      ┆ num   ┆ OPTIMAL ┆ 7      ┆ 0.257562 ┆ 1.01    │
│ avgmaxdpdlast9m_3716943P       ┆ num   ┆ OPTIMAL ┆ 2      ┆ 0.156351 ┆ 0.28    │
│ avgdbddpdlast3m_4187120P       ┆ num   ┆ OPTIMAL ┆ 5      ┆ 0.149264 ┆ 0.45    │
│ amtinstpaidbefduel24m_4187115A ┆ num   ┆ OPTIMAL ┆ 7      ┆ 0.099808 ┆ 1.83    │
│ …                              ┆ …     ┆ …       ┆ …      ┆ …  

In [3]:
import joblib

IV_MIN = cfg['champion']['iv_min']
print('IV_MIN =', IV_MIN)

BINS, IV = {}, {}
for w in ['B', 'A']:
    t0 = time.time()
    d = trB if w == 'B' else load_train('A')
    if w == 'B':
        del trB
    BINS[w], IV[w] = fit_many(d, CHAMP_FEATS[w])
    joblib.dump(BINS[w], MODEL_DIR / f'champion_binning_{w}.joblib')
    IV[w].write_csv(REPORTS / f'champion_iv_{w}.csv')
    print(f'{w}: {(time.time() - t0) / 60:.1f} phút | n = {d.height}'
          f' | không OPTIMAL: {IV[w].filter(pl.col("status") != "OPTIMAL").height}'
          f' | IV > {IV_MIN}: {IV[w].filter(pl.col("iv") > IV_MIN).height}'
          f' | IV > 0.5: {IV[w].filter(pl.col("iv") > 0.5).height}')
    del d

iv_cmp = (
    IV['A'].select('feature', pl.col('iv').alias('iv_A'))
    .join(IV['B'].select('feature', pl.col('iv').alias('iv_B')), on='feature', how='full', coalesce=True)
    .sort('iv_B', descending=True, nulls_last=True)
)
iv_cmp.write_csv(REPORTS / 'champion_iv_compare.csv')
print(iv_cmp.head(25))
print(IV['B'].group_by('dtype').agg(pl.len().alias('n'), (pl.col('iv') > IV_MIN).sum().alias('n_pass')))

IV_MIN = 0.02
B: 0.7 phút | n = 522720 | không OPTIMAL: 12 | IV > 0.02: 252 | IV > 0.5: 0
A: 1.0 phút | n = 841500 | không OPTIMAL: 12 | IV > 0.02: 192 | IV > 0.5: 0
shape: (25, 3)
┌─────────────────────────────────┬──────────┬──────────┐
│ feature                         ┆ iv_A     ┆ iv_B     │
│ ---                             ┆ ---      ┆ ---      │
│ str                             ┆ f64      ┆ f64      │
╞═════════════════════════════════╪══════════╪══════════╡
│ credit_bureau_a_2__pmts_dpd_30… ┆ null     ┆ 0.390113 │
│ credit_bureau_a_1__dpdmax_757P… ┆ null     ┆ 0.382076 │
│ credit_bureau_a_1__numberofove… ┆ null     ┆ 0.375148 │
│ credit_bureau_a_2__pmts_overdu… ┆ null     ┆ 0.361406 │
│ credit_bureau_a_1__overdueamou… ┆ null     ┆ 0.339538 │
│ …                               ┆ …        ┆ …        │
│ credit_bureau_a_2__pmts_overdu… ┆ null     ┆ 0.259159 │
│ avgdbddpdlast24m_3658932P       ┆ 0.227394 ┆ 0.257562 │
│ credit_bureau_a_2__pmts_overdu… ┆ 0.245063 ┆ 0.254811 │
│ credi

In [ ]:
bad = (
    pl.concat([IV[w].with_columns(pl.lit(w).alias('window')) for w in ['A', 'B']])
    .filter(pl.col('status') != 'OPTIMAL')
    .sort(['feature', 'window'])
)
print(bad.select('window', 'feature', 'dtype', 'status', 'n_bins', 'iv', 'seconds'))
print(bad.group_by('window', 'status').len())

In [4]:
CORR_MAX = 0.7

def load_rows(feats, cond):
    return lf.filter(cond).select(['WEEK_NUM', 'target'] + feats).collect(engine='streaming')

def woe_matrix(bins, feats, d):
    cols = []
    for c in feats:
        s = d.get_column(c)
        x = s.to_numpy() if s.dtype == pl.String else s.cast(pl.Float64).to_numpy()
        cols.append(bins[c].transform(x, metric='woe', metric_missing='empirical',
                                      metric_special='empirical').astype(np.float32))
    return np.column_stack(cols)

DATA = {}
for w in ['B', 'A']:
    t0 = time.time()
    sel = IV[w].filter(pl.col('iv') > IV_MIN).sort('iv', descending=True)['feature'].to_list()
    w0, w1 = WIN[w]
    tr = load_rows(sel, (pl.col('split') == 'train') & pl.col('WEEK_NUM').is_between(w0, w1))
    va = load_rows(sel, pl.col('split') == 'valid')
    Xtr, Xva = woe_matrix(BINS[w], sel, tr), woe_matrix(BINS[w], sel, va)

    C = np.abs(np.corrcoef(Xtr, rowvar=False))
    keep = []
    for i in range(len(sel)):
        if all(C[i, j] < CORR_MAX for j in keep):
            keep.append(i)

    DATA[w] = dict(
        feats=[sel[i] for i in keep],
        Xtr=Xtr[:, keep], ytr=tr['target'].to_numpy(),
        Xva=Xva[:, keep], yva=va['target'].to_numpy(), wk_va=va['WEEK_NUM'].to_numpy(),
    )
    print(f'{w}: IV > {IV_MIN}: {len(sel)} → sau lọc tương quan: {len(keep)}'
          f' | Train {Xtr.shape} | Valid {Xva.shape} | {(time.time() - t0) / 60:.1f} phút')
    del tr, va, Xtr, Xva, C

(REPORTS / 'champion_selected_after_corr.json').write_text(
    json.dumps({w: DATA[w]['feats'] for w in DATA}, indent=1), encoding='utf-8')

B: IV > 0.02: 252 → sau lọc tương quan: 79 | Train (522720, 252) | Valid (224421, 252) | 0.4 phút
A: IV > 0.02: 192 → sau lọc tương quan: 72 | Train (841500, 192) | Valid (224421, 192) | 0.5 phút


5507

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def gini(y, p):
    return 2 * roc_auc_score(y, p) - 1

def psi(expected, actual, n_bins=10):
    qs = np.unique(np.quantile(expected, np.linspace(0, 1, n_bins + 1)))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(expected, qs)[0] / len(expected), 1e-6, None)
    a = np.clip(np.histogram(actual, qs)[0] / len(actual), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

def fit_logit(X, y):
    return LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000).fit(X, y)

CHAMP, rows = {}, []
for w in ['B', 'A']:
    D, t0 = DATA[w], time.time()
    feats = D['feats']                                      # đã sắp theo IV giảm dần
    exp_sign = np.sign(np.corrcoef(D['Xtr'][:, 0], D['ytr'])[0, 1])
    idx, dropped = list(range(len(feats))), []
    while True:
        m = fit_logit(D['Xtr'][:, idx], D['ytr'])
        wrong = [k for k, b in enumerate(m.coef_[0]) if np.sign(b) != exp_sign]
        if not wrong:
            break
        k = max(wrong)                                      # sai dấu, IV thấp nhất
        dropped.append(feats[idx[k]])
        idx.pop(k)

    p_tr = m.predict_proba(D['Xtr'][:, idx])[:, 1]
    p_va = m.predict_proba(D['Xva'][:, idx])[:, 1]
    wk = D['wk_va']
    g_wk = [gini(D['yva'][wk == u], p_va[wk == u]) for u in np.unique(wk)]
    CHAMP[w] = dict(model=m, feats=[feats[i] for i in idx], dropped_sign=dropped)
    joblib.dump(CHAMP[w], MODEL_DIR / f'champion_logit_{w}.joblib')

    (pl.DataFrame({'feature': CHAMP[w]['feats'], 'coef': m.coef_[0]})
       .join(IV[w].select('feature', 'iv'), on='feature')
       .sort('iv', descending=True)
       .write_csv(REPORTS / f'champion_coef_{w}.csv'))

    rows.append(dict(
        window=w, exp_sign=int(exp_sign), n_feat=len(idx), n_dropped_sign=len(dropped),
        gini_train=gini(D['ytr'], p_tr), gini_valid=gini(D['yva'], p_va),
        gap=gini(D['ytr'], p_tr) - gini(D['yva'], p_va), psi_score=psi(p_tr, p_va),
        week_mean=float(np.mean(g_wk)), week_std=float(np.std(g_wk)), week_min=float(np.min(g_wk)),
        minutes=round((time.time() - t0) / 60, 1),
    ))
    print(w, 'xong', rows[-1]['minutes'], 'phút | bỏ do sai dấu:', len(dropped))

res = pl.DataFrame(rows)
res.write_csv(REPORTS / 'champion_window_ab.csv')
pl.Config.set_tbl_cols(20)
print(res)

B xong 0.9 phút | bỏ do sai dấu: 17
A xong 1.4 phút | bỏ do sai dấu: 15
shape: (2, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ window ┆ exp_si ┆ n_feat ┆ n_dro ┆ gini_ ┆ gini_ ┆ gap   ┆ psi_s ┆ week_ ┆ week_ ┆ week_ ┆ minut │
│ ---    ┆ gn     ┆ ---    ┆ pped_ ┆ train ┆ valid ┆ ---   ┆ core  ┆ mean  ┆ std   ┆ min   ┆ es    │
│ str    ┆ ---    ┆ i64    ┆ sign  ┆ ---   ┆ ---   ┆ f64   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│        ┆ i64    ┆        ┆ ---   ┆ f64   ┆ f64   ┆       ┆ f64   ┆ f64   ┆ f64   ┆ f64   ┆ f64   │
│        ┆        ┆        ┆ i64   ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       │
╞════════╪════════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ B      ┆ -1     ┆ 62     ┆ 17    ┆ 0.632 ┆ 0.617 ┆ 0.015 ┆ 0.003 ┆ 0.615 ┆ 0.026 ┆ 0.581 ┆ 0.9   │
│        ┆        ┆        ┆       ┆ 986   ┆ 459   ┆ 526   ┆ 095   ┆ 277   ┆ 714   ┆ 453   ┆       │
│ A 

In [6]:
pv = {}
for w in ['A', 'B']:
    cols = [DATA[w]['feats'].index(f) for f in CHAMP[w]['feats']]
    pv[w] = CHAMP[w]['model'].predict_proba(DATA[w]['Xva'][:, cols])[:, 1]

wk, y = DATA['B']['wk_va'], DATA['B']['yva']
assert np.array_equal(wk, DATA['A']['wk_va']) and np.array_equal(y, DATA['A']['yva'])

wkdf = pl.DataFrame([
    dict(week=int(u), gini_B=gini(y[wk == u], pv['B'][wk == u]), gini_A=gini(y[wk == u], pv['A'][wk == u]))
    for u in np.unique(wk)
]).with_columns((pl.col('gini_B') - pl.col('gini_A')).alias('diff'))
print(wkdf)
print('Số tuần B > A:', int((wkdf['diff'] > 0).sum()), '/', wkdf.height)

rng = np.random.default_rng(SEED)
diffs = []
for _ in range(200):
    i = rng.integers(0, len(y), len(y))
    diffs.append(gini(y[i], pv['B'][i]) - gini(y[i], pv['A'][i]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'ΔGini Valid (B − A) = {gini(y, pv["B"]) - gini(y, pv["A"]):.4f} | CI95 bootstrap [{lo:.4f}, {hi:.4f}]')

shape: (8, 4)
┌──────┬──────────┬──────────┬──────────┐
│ week ┆ gini_B   ┆ gini_A   ┆ diff     │
│ ---  ┆ ---      ┆ ---      ┆ ---      │
│ i64  ┆ f64      ┆ f64      ┆ f64      │
╞══════╪══════════╪══════════╪══════════╡
│ 44   ┆ 0.646407 ┆ 0.637799 ┆ 0.008608 │
│ 45   ┆ 0.59838  ┆ 0.591042 ┆ 0.007338 │
│ 46   ┆ 0.592395 ┆ 0.583892 ┆ 0.008503 │
│ 47   ┆ 0.656414 ┆ 0.652381 ┆ 0.004032 │
│ 48   ┆ 0.589604 ┆ 0.588069 ┆ 0.001535 │
│ 49   ┆ 0.622841 ┆ 0.61457  ┆ 0.008271 │
│ 50   ┆ 0.581453 ┆ 0.569353 ┆ 0.0121   │
│ 51   ┆ 0.634726 ┆ 0.629128 ┆ 0.005598 │
└──────┴──────────┴──────────┴──────────┘
Số tuần B > A: 8 / 8
ΔGini Valid (B − A) = 0.0072 | CI95 bootstrap [0.0052, 0.0099]


In [8]:
def score_cand(Xs, ys, sel, j):
    idx = sel + [j]
    m = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000).fit(Xs[:, idx], ys)
    if np.any(m.coef_[0] >= 0):                       # có hệ số sai dấu → loại ứng viên
        return j, -np.inf
    return j, gini(ys, m.decision_function(Xs[:, idx]))

D, fB = DATA['B'], CHAMP['B']['feats']
EXCLUDE = [c for c in FEATS if 'sex' in c.lower()]   # R7: thuộc tính giới tính
print('R7 loại khỏi toàn bộ 628 feature:', EXCLUDE)
fB = [f for f in fB if f not in EXCLUDE]
print('Ứng viên forward selection:', len(fB))
cols = [D['feats'].index(f) for f in fB]
Xtr, Xva, ytr, yva = D['Xtr'][:, cols], D['Xva'][:, cols], D['ytr'], D['yva']

rng = np.random.default_rng(SEED)
sub = rng.choice(len(ytr), 150_000, replace=False)
Xs, ys = Xtr[sub], ytr[sub]

MAX_K, TOL = 40, 0.005
GINI_FULL = gini(yva, pv['B'])
print(f'Valid Gini bản 62 feature: {GINI_FULL:.4f} → ngưỡng: {GINI_FULL - TOL:.4f}')

sel, path, t0 = [], [], time.time()
while len(sel) < MAX_K:
    cand = [j for j in range(len(fB)) if j not in sel]
    out = Parallel(n_jobs=N_JOBS)(delayed(score_cand)(Xs, ys, sel, j) for j in cand)
    j, g = max(out, key=lambda t: t[1])
    if not np.isfinite(g):
        print('Không còn ứng viên thỏa ràng buộc dấu')
        break
    sel.append(j)
    m = fit_logit(Xtr[:, sel], ytr)
    gv = gini(yva, m.decision_function(Xva[:, sel]))
    path.append(dict(k=len(sel), feature=fB[j], gini_train_sub=g, gini_valid=gv,
                     all_neg=bool(np.all(m.coef_[0] < 0))))
    print(f'{len(sel):>2} | {gv:.4f} | {fB[j]}')

path_df = pl.DataFrame(path)
path_df.write_csv(REPORTS / 'champion_forward_path_B.csv')
ok = path_df.filter((pl.col('gini_valid') >= GINI_FULL - TOL) & pl.col('all_neg'))
K_STAR = ok['k'].min() if ok.height else None
print(f'{(time.time() - t0) / 60:.1f} phút | k* = {K_STAR}')

R7 loại khỏi toàn bộ 628 feature: ['lastotherlnsexpense_631A', 'person_1__app__sex_738L']
Ứng viên forward selection: 61
Valid Gini bản 62 feature: 0.6175 → ngưỡng: 0.6125
 1 | 0.3297 | credit_bureau_a_2__pmts_dpd_303P__mean
 2 | 0.4030 | credit_bureau_a_2__pmts_dpd_1073P__mean
 3 | 0.4394 | days120_123L
 4 | 0.4729 | credit_bureau_a_1__totalamount_6A__sum
 5 | 0.5036 | maxdpdlast6m_474P
 6 | 0.5158 | lastst_736L
 7 | 0.5273 | tax__count
 8 | 0.5363 | avgdpdtolclosure24_3658938P
 9 | 0.5469 | pmtnum_254L
10 | 0.5488 | person_1__app__birth_259D
11 | 0.5537 | credit_bureau_a_1__dateofcredstart_739D__min
12 | 0.5594 | mobilephncnt_593L
13 | 0.5665 | person_1__app__education_927M
14 | 0.5717 | numinstpaidearly3d_3546850L
15 | 0.5775 | credit_bureau_a_1__overdueamountmax2_14A__mean
16 | 0.5804 | credit_bureau_a_2__pmts_dpd_303P__max_l12m
17 | 0.5847 | credit_bureau_a_1__prolongationcount_599L__max
18 | 0.5853 | credit_bureau_a_1__count
19 | 0.5886 | applprev_1__pmtnum_8L__mean
20 | 0.5916 |

In [9]:
import re

EXCLUDE = [c for c in FEATS if re.search(r'(^|_)sex(_|$)', c)]   # R7: khớp đúng từ "sex"
print('R7:', EXCLUDE)

m61 = fit_logit(Xtr, ytr)            # Xtr, Xva lúc này là 61 feature (đã bỏ giới tính)
G61 = gini(yva, m61.decision_function(Xva))
print(f'Bản 62 có giới tính: {GINI_FULL:.4f} | bản 61 không giới tính: {G61:.4f}'
      f' | chi phí công bằng: {GINI_FULL - G61:.4f}'
      f' | mọi hệ số âm: {bool(np.all(m61.coef_[0] < 0))}')

ok = path_df.filter((pl.col('gini_valid') >= G61 - TOL) & pl.col('all_neg'))
K_STAR = ok['k'].min() if ok.height else None
print(f'Ngưỡng mới: {G61 - TOL:.4f} | k* = {K_STAR}')

R7: ['person_1__app__sex_738L']
Bản 62 có giới tính: 0.6175 | bản 61 không giới tính: 0.6067 | chi phí công bằng: 0.0108 | mọi hệ số âm: True
Ngưỡng mới: 0.6017 | k* = 28


In [10]:
fB_sel = path_df['feature'].to_list()[:K_STAR]
cols = [DATA['B']['feats'].index(f) for f in fB_sel]
m = fit_logit(DATA['B']['Xtr'][:, cols], DATA['B']['ytr'])
assert np.all(m.coef_[0] < 0), 'Có hệ số sai dấu'

joblib.dump(CHAMP['B'], MODEL_DIR / 'champion_logit_B_full62.joblib')   # giữ bản 62 để đối chiếu
CHAMP['B'] = dict(model=m, feats=fB_sel, k_star=K_STAR, excluded_r7=EXCLUDE)
joblib.dump(CHAMP['B'], MODEL_DIR / 'champion_logit_B.joblib')
print(f'Champion B: {len(fB_sel)} feature | Valid Gini = '
      f'{gini(DATA["B"]["yva"], m.decision_function(DATA["B"]["Xva"][:, cols])):.4f}')

Champion B: 28 feature | Valid Gini = 0.6020


In [13]:
import math
from sklearn.metrics import roc_curve

PDO = cfg['champion']['scorecard']['pdo']
SCORE_REF = cfg['champion']['scorecard']['score_ref']
ODDS_REF = cfg['champion']['scorecard']['odds_ref']
FACTOR = PDO / math.log(2)
OFFSET = SCORE_REF - FACTOR * math.log(ODDS_REF)
print(f'PDO={PDO}, score_ref={SCORE_REF}, odds_ref={ODDS_REF} | Factor={FACTOR:.4f} | Offset={OFFSET:.4f}')

mB, fB = CHAMP['B']['model'], CHAMP['B']['feats']
b0, n = float(mB.intercept_[0]), len(fB)

# Bảng điểm từng bin
rows = []
for f, b in zip(fB, mB.coef_[0]):
    bt = BINS['B'][f].binning_table.build()
    bt = bt[bt.index != 'Totals']
    for _, r in bt.iterrows():
        woe = float(r['WoE'])
        rows.append(dict(feature=f, bin=str(r['Bin']), count=int(r['Count']),
                         event_rate=float(r['Event rate']), woe=woe, coef=float(b),
                         points=-(b * woe + b0 / n) * FACTOR + OFFSET / n))
card = pl.DataFrame(rows)
card.write_csv(REPORTS / 'champion_scorecard_B.csv')
print('Scorecard:', card.height, 'dòng (bin) cho', n, 'feature')

# Kiểm tra: WOE dùng khi train phải là WOE của các bin trong bảng điểm
cols = [DATA['B']['feats'].index(f) for f in fB]
Xtr_c, Xva_c = DATA['B']['Xtr'][:, cols], DATA['B']['Xva'][:, cols]
bad = [f for k, f in enumerate(fB)
       if not np.all(np.isclose(np.unique(Xva_c[:, k])[:, None],
                                card.filter(pl.col('feature') == f)['woe'].to_numpy()[None, :],
                                atol=1e-4).any(axis=1))]
print('Feature có WOE không khớp bảng điểm:', len(bad))

# Điểm của từng hồ sơ
s_tr = OFFSET - FACTOR * mB.decision_function(Xtr_c)
s_va = OFFSET - FACTOR * mB.decision_function(Xva_c)
y_tr, y_va = DATA['B']['ytr'], DATA['B']['yva']
print(pl.DataFrame({'train': np.percentile(s_tr, [0, 1, 10, 50, 90, 99, 100]),
                    'valid': np.percentile(s_va, [0, 1, 10, 50, 90, 99, 100])})
        .with_columns(pl.Series('pct', [0, 1, 10, 50, 90, 99, 100])))

# KS trên Valid (dùng −score vì score cao = rủi ro thấp)
fpr, tpr, _ = roc_curve(y_va, -s_va)
print(f'Valid: Gini = {gini(y_va, -s_va):.4f} | KS = {np.max(tpr - fpr):.4f}')

# Score band theo thập phân vị điểm trên Train
EDGES = np.quantile(s_tr, np.linspace(0, 1, 11))[1:-1]
def band_table(s, y):
    return (pl.DataFrame({'band': np.searchsorted(EDGES, s, side='right') + 1, 'y': y})
              .group_by('band')
              .agg(pl.len().alias('n'), pl.col('y').mean().alias('default_rate'))
              .sort('band'))
lo = np.concatenate([[-np.inf], EDGES])
hi = np.concatenate([EDGES, [np.inf]])
bands = (band_table(s_tr, y_tr)
         .join(band_table(s_va, y_va), on='band', suffix='_valid')
         .with_columns(pl.Series('score_from', np.round(lo, 1)), pl.Series('score_to', np.round(hi, 1))))
bands.write_csv(REPORTS / 'champion_score_bands_B.csv')
(REPORTS / 'champion_scorecard_params.json').write_text(json.dumps(dict(
    pdo=PDO, score_ref=SCORE_REF, odds_ref=ODDS_REF, factor=FACTOR, offset=OFFSET,
    intercept=b0, n_features=n, features=fB, band_edges=EDGES.tolist()), indent=1), encoding='utf-8')
print(bands)
for c in ['default_rate', 'default_rate_valid']:
    print(c, '| giảm đơn điệu theo band:', bool(np.all(np.diff(bands[c].to_numpy()) < 0)))

PDO=20, score_ref=600, odds_ref=50 | Factor=28.8539 | Offset=487.1229
Scorecard: 212 dòng (bin) cho 28 feature
Feature có WOE không khớp bảng điểm: 0
shape: (7, 3)
┌────────────┬────────────┬─────┐
│ train      ┆ valid      ┆ pct │
│ ---        ┆ ---        ┆ --- │
│ f64        ┆ f64        ┆ i64 │
╞════════════╪════════════╪═════╡
│ 427.335144 ┆ 440.580597 ┆ 0   │
│ 523.559651 ┆ 523.788232 ┆ 1   │
│ 560.805127 ┆ 562.244629 ┆ 10  │
│ 606.230347 ┆ 608.203491 ┆ 50  │
│ 646.809937 ┆ 648.800781 ┆ 90  │
│ 673.392133 ┆ 675.223254 ┆ 99  │
│ 714.878601 ┆ 715.054382 ┆ 100 │
└────────────┴────────────┴─────┘
Valid: Gini = 0.6020 | KS = 0.4557
shape: (10, 7)
┌──────┬───────┬──────────────┬─────────┬────────────────────┬────────────┬──────────┐
│ band ┆ n     ┆ default_rate ┆ n_valid ┆ default_rate_valid ┆ score_from ┆ score_to │
│ ---  ┆ ---   ┆ ---          ┆ ---     ┆ ---                ┆ ---        ┆ ---      │
│ i64  ┆ u32   ┆ f64          ┆ u32     ┆ f64                ┆ f64        ┆ f64    